# QueryRouter++ Exploration Notebook

**description:** Interactive exploration of model data, feature extraction, compatibility scoring, and routing strategies.  
**agent:** coder  
**date:** 2026-03-24  
**version:** 1.0

## Cell 1: Load data matrices and show statistics

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np

# Add project root to path
project_root = Path(".").resolve().parent
sys.path.insert(0, str(project_root))

data_dir = project_root / "data"

# Load CSVs
df_bench = pd.read_csv(data_dir / "models_benchmark_matrix.csv", comment="#")
df_cost = pd.read_csv(data_dir / "models_cost_matrix.csv", comment="#")
df_eco = pd.read_csv(data_dir / "models_eco_matrix.csv", comment="#")

print(f"Benchmark matrix: {df_bench.shape[0]} models x {df_bench.shape[1]} columns")
print(f"Cost matrix: {df_cost.shape[0]} models x {df_cost.shape[1]} columns")
print(f"Eco matrix: {df_eco.shape[0]} models x {df_eco.shape[1]} columns")
print()
print("=== Benchmark Statistics ===")
print(df_bench.describe())
print()
print("=== Cost Statistics ===")
print(df_cost[["model_id", "input_price_per_1m_tokens_usd", "output_price_per_1m_tokens_usd"]].to_string(index=False))

## Cell 2: Visualize benchmark scores heatmap

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Prepare benchmark heatmap data
bench_cols = ["mmlu_score", "humaneval_score", "gsm8k_score", "math_score", "hellaswag_score", "arc_score"]
bench_labels = ["MMLU", "HumanEval", "GSM8K", "MATH", "HellaSwag", "ARC"]

heatmap_data = df_bench.set_index("model_id")[bench_cols].astype(float)
heatmap_data.columns = bench_labels

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(
    heatmap_data,
    annot=True,
    fmt=".3f",
    cmap="YlOrRd",
    vmin=0.5,
    vmax=1.0,
    ax=ax,
    linewidths=0.5,
)
ax.set_title("LLM Benchmark Scores Heatmap")
ax.set_ylabel("Model")
plt.tight_layout()
plt.show()

## Cell 3: Demo QueryFeaturizer on 5 example queries

In [ ]:
from queryrouter.core.query_featurizer import QueryFeaturizer

featurizer = QueryFeaturizer()

example_queries = [
    "Write a Python function that merges two sorted lists",
    "Solve: If f(x) = 3x^2 - 2x + 1, find f'(x)",
    "Write a short story about a robot discovering emotions",
    "What is the capital of Burkina Faso?",
    "Summarize this article in three bullet points",
]

feature_names = [
    "avg_word_len", "sentence_count", "word_count",
    *[f"dom_{d}" for d in featurizer.DOMAINS],
    *[f"task_{t}" for t in featurizer.TASK_TYPES],
    "output_len", "reasoning_depth", "language",
    "code_presence", "math_presence", "creativity", "factual_precision",
]

for q in example_queries:
    features = featurizer.featurize(q)
    print(f"\nQuery: {q}")
    print(f"  Shape: {features.shape}")
    # Show non-zero features
    for name, val in zip(feature_names, features):
        if val > 0.01:
            print(f"  {name}: {val:.3f}")

## Cell 4: Demo CompatibilityScorer with different weight vectors

In [ ]:
from queryrouter.core.model_registry import ModelRegistry
from queryrouter.core.compatibility_scorer import CompatibilityScorer, WeightVector
from queryrouter.data.normalizers import FeatureNormalizer

registry = ModelRegistry(data_dir)
all_models = registry.get_all()

normalizer = FeatureNormalizer()
normalizer.fit(all_models)
scorer = CompatibilityScorer(normalizer)

query = "Write a Python function to sort a list efficiently"
features = featurizer.featurize(query)

weight_profiles = {
    "Performance": WeightVector(0.7, 0.1, 0.1, 0.1),
    "Cost": WeightVector(0.1, 0.7, 0.1, 0.1),
    "Balanced": WeightVector(0.25, 0.25, 0.25, 0.25),
    "Ecology": WeightVector(0.1, 0.1, 0.1, 0.7),
}

for profile_name, weights in weight_profiles.items():
    print(f"\n=== {profile_name} Profile ===")
    results = scorer.score_all(features, all_models, weights)
    for ms in results[:3]:
        print(f"  {ms.model_id}: {ms.score:.4f} (P={ms.breakdown['performance']:.3f}, K={ms.breakdown['cost']:.3f}, L={ms.breakdown['latency']:.3f}, E={ms.breakdown['ecology']:.3f})")

## Cell 5: Demo full QueryRouter with all 3 strategies

In [ ]:
from queryrouter.core.router import QueryRouter
from queryrouter.api.schemas import RoutingRequest, UserPreferences

test_queries = [
    ("Write a Python quicksort", "cost_performance"),
    ("What is the capital of France?", "cost"),
    ("Prove that sqrt(2) is irrational", "performance"),
]

for strategy in ["direct", "cascade", "embedding"]:
    print(f"\n{'='*60}")
    print(f"Strategy: {strategy.upper()}")
    print(f"{'='*60}")
    router = QueryRouter(strategy=strategy, data_dir=data_dir)
    
    for query_text, opt_mode in test_queries:
        request = RoutingRequest(
            query=query_text,
            preferences=UserPreferences(optimize_for=opt_mode),
        )
        response = router.route(request)
        print(f"\n  Query: {query_text} (optimize={opt_mode})")
        print(f"  -> {response.recommended_model} (score={response.scores[0].score:.4f}, cost=${response.estimated_cost_usd:.6f})")

## Cell 6: Pareto frontier plot (cost vs performance)

In [ ]:
# Compute performance and cost scores for all models
balanced_weights = WeightVector(0.25, 0.25, 0.25, 0.25)
test_query = "Explain machine learning in detail"
test_features = featurizer.featurize(test_query)

perf_scores = []
cost_scores = []
model_names = []

for m in all_models:
    ms = scorer.score(test_features, m, balanced_weights)
    perf_scores.append(ms.breakdown["performance"])
    cost_scores.append(m.cost_input_per_1m + m.cost_output_per_1m)
    model_names.append(m.model_id)

fig, ax = plt.subplots(figsize=(10, 7))
scatter = ax.scatter(cost_scores, perf_scores, s=100, c='steelblue', edgecolors='black', zorder=5)

for i, name in enumerate(model_names):
    ax.annotate(name, (cost_scores[i], perf_scores[i]),
                textcoords="offset points", xytext=(8, 5), fontsize=8)

# Draw Pareto frontier
points = sorted(zip(cost_scores, perf_scores, model_names), key=lambda x: x[0])
pareto_costs, pareto_perfs = [], []
max_perf = -1
for c, p, n in points:
    if p > max_perf:
        pareto_costs.append(c)
        pareto_perfs.append(p)
        max_perf = p

ax.plot(pareto_costs, pareto_perfs, 'r--', linewidth=2, label='Pareto Frontier', zorder=4)

ax.set_xlabel("Total Cost (USD per 1M tokens)", fontsize=12)
ax.set_ylabel("Performance Score", fontsize=12)
ax.set_title("Cost vs Performance: Pareto Frontier", fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()